# Chain

**Chain**(체인)은 여러 컴포넌트(요소)를 정해진 순서대로 연결하여 **복잡한 AI 작업을 '단계별'로 자동화**할 수 있도록 돕는 구조이다.

- 각 컴포넌트는 **이전 처리결과를 입력으로 받아 처리한 후 다음 단계로 결과를 전달**한다.
- 복잡한 작업을 여러 개의 단순한 단계로 나누고, 각 단계를 '**순차적**'으로 실행함으로써 전체 작업을 체계적으로 구성할 수 있다.
  - (순차적 : 장점이자 한계. 일은 순차적이지만은 않음. 반복, 제어문 등, 앞 결과에 따라 뒤 흐름이 바뀌어야 하는 경우도 있음. => lang그래프)

## 기본 개념

- 체인은 하나의 LLM 호출에 그치지 않고 **여러 LLM 호출이나 도구 실행등을 순차적으로 연결**하여 실행 할 수 있다.
- 예를 들어, 사용자의 '질문 → 검색 → 요약 → 응답 생성' 같은 일련의 작업을 체인으로 구성할 수 있다.('분업화')
- 이러한 체인구조를 사용하면 **작업흐름이 명확**해지고 **코드의 재사용성**이 높아지며 **유지 보수 및 확장성이** 향상된다.

## LangChain에서의 Chain 구성 방식

LangChain은 다음 두 가지 방식을 통해 체인을 구성할 수 있다.

### 1. Off-the-shelf Chains 방식 (클래식 방식)

- LangChain에서 제공하는 **미리 정의된 Chain 클래스**(예: `LLMChain`, `SequentialChain`, `SimpleSequentialChain`)를 활용하는 방식이다.
- 각 클래스는 다양한 chain 알고리즘들을 미리 구현한 것으로 상황에 맞는 것을 선택하여 필요한 구성요소를 전달해 생생한다.
- 이 방식은 LangChain의 **초기 방식**이며, 새로운 기능 확장이나 유연한 구성에 한계가 있기 때문에 현재 **더 이상 사용되지 않음(deprecated)** 상태이다.
  - 현재 LangChain에서는 권장하지 않는 방식이다.

### 2. LCEL (LangChain Expression Language) 방식

- LCEL은 체인을 '표현식(Expression) 기반'의 선언적 파이프라인 방식으로 구성할 수 있도록 설계된 최신 체인 구성 방법이다. 
- 각 '컴포넌트들을 `|` 연산자로 연결'하여, 흐름이 자연스럽게 이어지는 형태의 체인을 구성한다.
- LCEL 방식은 간결하고 선언적인 문법을 제공하여 **직관적이고 융통성과 확장성 있는 체인 구성**이 가능하다.
- langchain classic
- LCEL은
  - '선형적 흐름 구조'를 가진다.
  - 문법이 간결하고 선언적이다.
  - 체인의 구조가 코드만 봐도 쉽게 파악된다.
  - 유연하고 확장성이 매우 뛰어나다.
- '`Runnable` 기반' 구조
  - LCEL방식을 구성하는 모든 컴포넌트들은 `Runnable` 이라는 공통 인터페이스를 기반으로 동작한다.
  - 체인을 구성하는 각 '컴포넌트들은 `Runnable` 을 상속'(모든 컴포넌트가 Runnable 타입.)하여 구현하여 이를 통해 일관된 실행 인터페이스를 제공한다.
  - **공통 메소드**:
    - `invoke()`: 단일 입력에 대한 처리
    - `batch()`: 다수 입력을 묶어서 한번에 처리
    - `stream()`: 스트리밍 방식의 요청
    - `ainvoke()`, `abatch()`, `astream()`: 비동기적 처리 메소드

In [2]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
prompt = ChatPromptTemplate.from_template(
    template="{item}에 어울리는 브랜드 이름 {count}개를 만들어 주세요."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [5]:
query = prompt.invoke({"item":"가방", "count":5})
res = model.invoke(query)
result = parser.invoke(res)
print(result)

물론이죠! 가방에 어울리는 **브랜드 이름 5개** 만들어 드릴게요.

1. **루미노백(LuminoBAG)**  
2. **노바포트(NovaPort)**  
3. **시엘라크(Siellac)**  
4. **아르덴트백(ArdentBag)**  
5. **클리어런트(ClearLant)**


In [6]:
############################################################
# 기존의 Off the shell 방식 - langchain-classic 설치 필요
############################################################
from langchain_classic import LLMChain
# chain을 구성하는 요소들을 넣어서 생성.
# prompt_template -
chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=parser
)

res = chain.invoke({"item":"가방", "count":3})
print(res)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_15212\1295254559.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


{'item': '가방', 'count': 3, 'text': '1. **루나바그**  \n2. **모먼트케이**  \n3. **엘리노트**'}


In [7]:
##############################
#  LCEL
##############################
chain2 = prompt | model | parser
print(type(chain2))
res2 = chain2.invoke({"item":"TV 브랜드", "count":3}) 

<class 'langchain_core.runnables.base.RunnableSequence'>


In [8]:
res2

'1. **선명비전TV**  \n2. **블루라이트네오**  \n3. **프라임뷰 컬렉션**'

In [9]:
from langchain_core.runnables import Runnable
isinstance(model, Runnable), isinstance(prompt, Runnable), isinstance(parser, Runnable), isinstance(chain2, Runnable)

(True, True, True, True)

# Runnable 타입 주요 클래스


## [Runnable](https://reference.langchain.com/python/langchain_core/runnables/#langchain_core.runnables.base.Runnable)
- LangChain의 Runnable은 '실행 가능한 작업 단위를 캡슐화'한 개념으로, 데이터 흐름의 각 단계를 정의하고 **체인(chain) 에 포함 되어**  '복잡한 작업의 각 단계를 수행' 한다.
- **Chain을 구성하는 class들**은 Runnable의 상속 받아 구현한다.
- **Prompt Template클래스**, **Chat 모델**, **Output Parser 클래스** 등 다양한 컴포넌트가 Runnable을 상속받아 구현된다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 호출, 출력 파싱 등)을 수행하는 독립적인 컴포넌트이다.
    - 각 컴포넌트는 독립적으로 테스트 및 재사용이 가능하며, 조합하여 복잡한 체인을 구성할 수 있다.
- 체인 연결 및 작업 흐름 관리:
    - Runnable은 체인(chain, 일련의 연결된 작업 흐름)을 구성하는 기본 단위로 사용된다.
    - LangChain Expression Language(LCEL)를 사용하면 | 연산자를 통해 여러 Runnable을 쉽게 연결할 수 있다.
    - 입력과 출력의 형식을 일관되게 유지하여 각 단계가 자연스럽게 연결된다.
- 모듈화 및 디버깅 용이성:
    - 각 단계가 명확히 분리되어 문제 발생 시 어느 단계에서 오류가 발생했는지 쉽게 확인할 수 있다.
    - 복잡한 작업을 작은 단위로 나누어 체계적으로 관리할 수 있다.  

runnable이 가지고 있는 메소드 -  
공통적인 기능을 상위클래스에서 구현되어 있음 - 하위클래스에서는 (오버라이딩해서)  값만 변경하면 됨 
공통된 방식(ex. invoke)으로 호출될 수 있도록(상속 : 타입 통일 개념)
      
### Runnable의 표준 메소드
- 모든 Runnable이 구현하는 공통 메소드
    - **`invoke(input, config:RunnableConfig)->output`**: 단일 입력을 처리하여 결과를 반환.
    - **`batch(input:list, config:RunnableConfig|list[RunnableConfig]) -> list[Output]`**: 여러 입력 데이터들을 한 번에 처리.
    - **`stream(input, config:RunnableConfig) -> Iterator[Output]`**: 입력에 대해 스트리밍 방식으로 응답을 반환.
    - **`assign(**kwargs)`**:
      -  '앞 Runnable의 출력 결과'에 '새로운 key–value 쌍'의 'Field 추가(assign)' 하여 '다음 Runnable로 전달'.
      -  값으로는 Runnable 객체(LCEL체인등)나 고정 값(리터럴) 모두 가능하며, 각 항목은 실행 시 평가되어 기존 출력에 병합한다.
      -  주로 앞 단계의 출력에 '부가 정보(field)를 추가'하고자 할 때 사용한다. 특히 `RunnablePassthrough`와 결합해, 입력을 그대로 넘기면서 특정 field만 추가할 때 자주 사용


### Runnable의 주요 구현체(하위 클래스)

- 다음 클래스들은 기능을 제공하는 것이 아니라 **chain 구조를 다양하게 구성** 할 수 있도록 도와주는 **Runnable** 타입의 클래스들이다.

- **`RunnableSequence`**
    - 여러 `Runnable`을 순차적으로 연결하여 실행하는 구성이다.
    - 각 단계의 출력이 다음 단계의 입력으로 전달된다.
    - 보통은 LCEL 문법을 사용해서 정의한다.
      - 'LCEL을 사용하여 체인을 구성'할 경우 '자동으로 `RunnableSequence`로 변환'된다.


In [12]:
from langchain_core.runnables import RunnableSequence

chain = prompt | model | parser # (=) chain = RunnableSequence(prompt, model, parser) 
# chain

chain.invoke({"item":"물", "count":2})

'1) **물빛(무루빛)**  \n2) **맑수결(맑은수의 결)**'


- **`RunnableLambda`**
    - Lambda 표현식의 함수를 `Runnable`로 변환할 때 사용한다.    
    - 일반함수도 `RunnableLambda`로 변환할 수 있다. 단 일반 함수는 변환 없이 chain에 포함 시킬 수 있기 때문에 굳이 변환할 필요가 없다.
    - Runnable로 만들 함수 구문
        - parameter: 입력 값 1개선언.
        - return: '다음 chain에 전달할 값의 형식'(다음 chain 형식에 맞춰야 함)


In [14]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda(함수)
# 함수 - 파라미터(1개 -> 앞 chain으로부터 받을 값에 맞춤)
#      - 리턴값 -> 다음 chain의 입력 type에 맞게 반환(dict, str, tuple 등..)
c1 = RunnableLambda(lambda input_data: f"{input_data}에 대해서 한 문장으로 설명해줘.")
# type(c1)
c1.invoke("LLM 모델")

'LLM 모델에 대해서 한 문장으로 설명해줘.'

In [15]:
chain = c1 | model | parser
chain.invoke("LLM 모델") # chain 호출 시에는 첫 번째 컴포넌트에 전달할 값을 넣어서 호출

'LLM(대규모 언어 모델)은 방대한 텍스트 데이터를 학습해 다음에 올 단어를 예측하는 방식으로 자연어를 이해하고 생성하는 인공지능 모델입니다.'

In [20]:
prompt = ChatPromptTemplate(
    messages=[
        ("system", "모든 응답은 100글자 이내로 작성해줘."),
        ("user", "{query}")
    ]
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

# Chain 구성. chain 응답 : LLM 응답 내용, 글자 수
chain = prompt | model | parser | RunnableLambda(lambda x: (x, len(x)))

In [21]:
res = chain.invoke("AI에 대해서 설명해줘.")
print(res)

('AI는 데이터를 학습해 인식·추론·생성하는 기술입니다. 인간의 일부 지능을 모방합니다.', 48)


In [22]:
def get_value_len(value:str):
    return value, len(value)

# 일반 함수를 chain의 구성으로 포함시킬 수 있음 -> 내부적으로 Runnable로 변환돼 들어감
chain2 = prompt | model | parser | get_value_len # RunnableLambda(get_value_len)
chain2.invoke("크리스마스")

('메리 크리스마스! 🎄', 11)


-  **`RunnablePassthrough`**
    - 입력 데이터를 가공하지 않고 그대로 다음 단계로 전달하는 `Runnable`이다.
      - 앞 Runnable으로 부터 전달 받은 **입력 값을 다음 Runnable로 그대로 전달**한다.
           - `RunnablePassthrough()`
      - 입력받은 값에 **Field를 추가**해서 전달할 경우 `assign()` 메소드를 사용한다.
           - `RunnablePassthrough.assign(new_key1="new_value1", new_key2="new_value2", ..)`


In [26]:
from langchain_core.runnables import RunnablePassthrough

rp = RunnablePassthrough() # 단순히 받은 값을 다음으로 통과시킴

result = rp.invoke("안녕하세요")
result = rp.invoke([1, 2, 3, 4, 5])
result = rp.invoke({"a":10, "b":20})

print(result)

{'a': 10, 'b': 20}


In [28]:
# 입력받은 값(딕셔너리)에 item 추가해 다음으로 전달
r1 = RunnableLambda(lambda x: "서울시 금천구 독산동")
r2 = RunnableLambda(lambda x: "010-1111-2222")

# assign(key=Runnable, ...)
## 입력받은 딕셔너리에 address, tel_no key 추가. value는 Runnable 호출해 반환된 값을 설정
rp2 = RunnablePassthrough.assign(
    address=r1,
    tel_no=r2
)
result = rp2.invoke({"name":"홍길동"})
result

{'name': '홍길동', 'address': '서울시 금천구 독산동', 'tel_no': '010-1111-2222'}


- **`RunnableParallel`**
    - 여러 `Runnable`을 '병렬'로 실행한 후, 결과를 결합하여 다음 단계로 전달한다.
    - 
        ```python
        RunnableParallel(
            {
                "key1":Runnable1, 
                "key2":Runnable2,
                "key3":Runnable3, ...
            }
        )
        ```
    - 각 Runnable의 실행결과를 Value로 Dictionary를 생성해서 반환한다.
    - LCEL로 정의할 때는 Chain에 dictionary로 정의한다.



In [30]:
from langchain_core.runnables import RunnableParallel

r1 = RunnableLambda(lambda x: x + 10)
r2 = RunnableLambda(lambda x: x - 10)
r3 = RunnableLambda(lambda x: x * 10)
r4 = RunnableLambda(lambda x: x / 10)

# c = r1 | r2 | r3 | r4
parallel = RunnableParallel(
    {
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value":RunnablePassthrough() # 입력받은 값을 그대로 다음으로 넘겨야 할 경우
    }
)

result = parallel.invoke(200)
result

{'value1': 210,
 'value2': 190,
 'value3': 2000,
 'value4': 20.0,
 'org_value': 200}

In [32]:
c = RunnablePassthrough() | {
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value":RunnablePassthrough()
    }

c.invoke(200)

{'value1': 210,
 'value2': 190,
 'value3': 2000,
 'value4': 20.0,
 'org_value': 200}

### LCEL Chain 예제

In [ ]:
##################################################
# TODO 1
# 음식 이름을 입력하면 그 음식의 레시피를 llm이 출력하는 Chain을 LCEL 을 이용해서 구성한다.
# 입력 : 음식 이름 - recipe_chain.invoke({"food":"김치찌개"})
# 출력 : 음식의 레시피 - 김치찌개 레시피. 

# chain구성: prompt_template -> model(gpt-5.4-mini) -> StrOutputParser

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

system_prompt = """
<instruction>
당신은 요리전문 AI Assistant입니다.
요청받은 음식의 레시피를 자세하고 쉽게 작성해 주세요.
출력 방법은 아래 output_format을 참고해서 응답해주세요.
</instruction>

<output_format>
- Markdown 형식으로 답변을 작성합니다.
- 응답 내용에는 다음 항목들을 포함합니다.
    - 요리 이름
    - 요리 기본 정보
        - 난이도
        - 조리시간
        - 인분
    - 요리에 필요한 재료(없을 시 대체 재료, 재료의 필요한 양 등)
    - 요리 방법
    - 팁
</output_format>
"""
parser = StrOutputParser()
model = ChatOpenAI(model="gpt-5.4-mini")
prompt = ChatPromptTemplate(
    messages=[
        ("system", system_prompt),
        ("user", "{food}의 레시피를 작성해 주세요.")
    ]
)

In [10]:
recipe_chain = prompt | model | parser

res = recipe_chain.invoke({"food":"김치찌개"})

In [11]:
from IPython.display import Markdown
# print(res)
Markdown(res)

# 김치찌개 레시피

## 요리 기본 정보
- **난이도:** 초급
- **조리시간:** 약 30~40분
- **인분:** 2~3인분

## 필요한 재료
### 주재료
- 묵은지 또는 잘 익은 김치 2컵
- 돼지고기(앞다리살, 목살, 삼겹살 중 택1) 200g  
  - **대체 재료:** 참치 1캔, 꽁치 1캔, 스팸 100g
- 두부 1/2모
- 양파 1/2개
- 대파 1대
- 청양고추 1~2개

### 국물 재료
- 물 또는 멸치육수 3컵
- 김치 국물 1/2컵
- 다진 마늘 1큰술
- 고춧가루 1큰술
- 국간장 1큰술
- 소금 약간
- 후추 약간

### 선택 재료
- 식용유 1큰술
- 설탕 1/2작은술  
  - **팁:** 김치가 너무 시면 설탕을 아주 조금 넣으면 맛이 부드러워집니다.

---

## 요리 방법
1. **재료 손질하기**  
   김치는 먹기 좋게 썰고, 돼지고기는 한입 크기로 준비합니다.  
   두부는 먹기 좋은 크기로 썰고, 양파는 채 썰거나 굵게 썰어줍니다.  
   대파와 청양고추는 어슷 썹니다.

2. **고기와 김치 볶기**  
   냄비에 식용유를 두르고 돼지고기를 먼저 넣어 중불에서 볶습니다.  
   고기가 반쯤 익으면 김치를 넣고 2~3분 정도 함께 볶아줍니다.  
   이 과정을 거치면 찌개의 맛이 훨씬 깊어집니다.

3. **양념 넣기**  
   다진 마늘, 고춧가루, 국간장을 넣고 고루 섞어 볶습니다.  
   김치 국물도 넣어주면 감칠맛이 더 살아납니다.

4. **국물 붓고 끓이기**  
   물 또는 멸치육수를 부은 뒤 센 불에서 끓입니다.  
   끓기 시작하면 중약불로 줄이고 10~15분 정도 끓여 김치와 고기의 맛이 잘 우러나도록 합니다.

5. **나머지 재료 넣기**  
   양파, 두부, 청양고추를 넣고 3~5분 더 끓입니다.  
   마지막에 대파를 넣고 한소끔 끓인 뒤 맛을 보고 필요하면 소금이나 후추로 간을 맞춥니다.

6. **완성하기**  
   불을 끄고 그릇에 담아내면 뜨끈한 김치찌개가 완성됩니다.  
   밥과 함께 먹으면 더욱 맛있습니다.

---

## 팁
- **묵은지**를 사용하면 훨씬 깊고 진한 맛이 납니다.
- **돼지고기를 먼저 볶는 것**이 국물 맛을 좋게 만드는 핵심입니다.
- 참치나 꽁치를 넣을 경우에는 **고기를 대신하여** 사용하고, 국물이 너무 짜지 않게 김치 국물 양을 조절하세요.
- 더 얼큰하게 먹고 싶다면 **고춧가루나 청양고추**를 조금 더 넣어도 좋습니다.
- 남은 김치찌개는 다음 날 더 맛있어지는 경우가 많습니다.

원하시면 제가 이어서 **“1인분 기준 레시피”** 또는 **“참치김치찌개 버전”**으로도 바꿔드릴게요.

In [ ]:
##############################################################
#  TODO 2
# 번역할 내용, 번역할 언어 를 입력하면 내용을 그 언어로 번역하는 Chain을 LCEL 을 이용해서 구성한다.
#
## 입력: 번역할 내용, 언어.  translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})
## 출력: "번역할 내용"을 "언어" 로 번역한 결과 - "How are you?".

# chain구성: prompt_template -> model(gpt-5.4-mini) -> StrOutputParser

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

system_prompt2="""
<instruction>
당신은 다국어가 가능한 숙련된 번역 AI Assistant입니다.
<input_data> 항목에 작성된 내용을 참고해서 **요청된 문서**의 내용을 **요청된 언어**로 번역해 주세요.
내용의 의미를 해치지 않는 범위에서 최대한 읽기 쉽게 작성해 주세요.
</instruction>

<input_data>
- 번역할 내용: {content}
- 번역할 언어: {language}
</input_data>
"""
prompt_trans = ChatPromptTemplate.from_template(
    template=system_prompt2
)
model_trans = ChatOpenAI(model="gpt-5.4-mini")
translate_chain = prompt_trans | model_trans | StrOutputParser()

In [13]:
res = translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})

In [14]:
print(res)

Hello.


In [15]:
content = """The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data."""

res = translate_chain.invoke({"content":content, "language":"한국어"})

In [16]:
print(res)

주요 시퀀스 변환 모델은 인코더와 디코더를 포함하는 복잡한 순환 신경망 또는 합성곱 신경망에 기반한다. 최고 성능을 보이는 모델들은 주의(attention) 메커니즘을 통해 인코더와 디코더를 연결하기도 한다. 우리는 순환과 합성곱을 완전히 배제하고 오직 주의 메커니즘에만 기반한 새로운 간단한 네트워크 구조, Transformer를 제안한다. 두 가지 기계 번역 작업에 대한 실험에서 이 모델들은 더 우수한 품질을 보였으며, 동시에 더 병렬화하기 쉬워 학습 시간도 크게 줄었다. 우리의 모델은 WMT 2014 영어-독일어 번역 과제에서 28.4 BLEU를 달성하여, 앙상블을 포함한 기존 최고 결과를 2 BLEU 이상 앞섰다. WMT 2014 영어-프랑스어 번역 과제에서는 8개의 GPU로 3.5일 동안 학습한 뒤 41.8의 BLEU 점수를 기록하며, 단일 모델 기준 새로운 최고 성능을 수립했다. 이는 문헌에 보고된 최고 모델들의 학습 비용에 비하면 매우 작은 수준이다. 또한 우리는 Transformer가 다른 과제에도 잘 일반화됨을 보였으며, 충분한 데이터와 제한된 데이터 모두에서 영어 구문 분석에 성공적으로 적용했다.


In [17]:
from langchain_core.runnables import Runnable

isinstance(translate_chain, Runnable) # -> True : 다른 chain에 넣어줄 수 있음

True

### Chain과 Chain간의 연결

In [21]:
from operator import itemgetter

ig = itemgetter("language") # 자료구조에서 값을 조회할 index or key를 넣고 객체 생성

a = {"language":"영어", "name":"홍길동"}
ig(a) # 자료구조 넣어주면 생성할 때 지정한 index or key의 값 조회해 반환

ig2 = itemgetter(2)
b = [10, 20, 30, 40, 50, 60]
ig2(b)

30

In [22]:
# 음식 레시피를 원하는 언어로 출력하는 AI Agent가 필요
## recipe_chain과 translate_chain을 연결
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

# {} : RunnableParallel
chain = {
    "content":recipe_chain,
    "language":itemgetter("language")
    # "language":RunnableLambda(lambda x: x['language'])
} | translate_chain

In [23]:
res = chain.invoke({"food":"김치찌개", "language":"독일어"})

In [24]:
print(res)

# Kimchi-Jjigae-Rezept

## Gerichtname
**Kimchi-Jjigae**

## Grundinformationen zum Gericht
- **Schwierigkeit:** Einfach
- **Zubereitungszeit:** ca. 30 Minuten
- **Portionen:** 2–3 Portionen

## Zutaten
- ca. **2 Tassen Sauerkimchi**
- **200 g Schweinefleisch** aus der Schulter oder vom Nacken  
  - falls nicht vorhanden, kann es durch **Thunfisch, Spam oder Tofu** ersetzt werden
- **1/2 Zwiebel**
- **1 Frühlingszwiebel**
- **1/2 Block Tofu**
- **1 EL gehackter Knoblauch**
- **1 EL Gochugaru (Koreanisches Chilipulver)**
- **1 EL Suppe-Sojasoße (Guk-ganjang)**  
  - falls nicht vorhanden, kann normale Sojasoße verwendet werden
- **1/2 Tasse Kimchi-Saft**
- **2–3 Tassen Wasser**
- **1 EL Speiseöl**
- **1 TL Zucker**  
  - verwenden, wenn das Kimchi zu sauer ist, um den Geschmack auszugleichen
- **etwas Pfeffer**
- Optionale Zutaten: **1–2 Cheongyang-Chilis**, **1 TL Thunfischsauce**

## Zubereitung
1. **Zutaten vorbereiten**  
   Das Kimchi in mundgerechte Stücke schneiden und das Schwei

## 함수를 Runnable로 정의하기

### 함수 구현
- **파라미터**
   - 이전 Chain에서 출력한 값을 입력으로 받을 수 있도록 정의한다.
- **리턴값**
   - 다음 Chain으로 입력할 값을 반환하도록 구현한다.

### Runnable 타입으로 만들기
1. LCEL Chain안에 함수를 구성요소로 포함시키면, 그 함수는 자동으로 `Runnable` 로 취급된다.
   - 별도의 래핑이나 추가 처리는 필요하지 않다.
2. `RunnableLambda()` 에 넣어 명시적으로 `Runnable` 타입으로 만든다.
   - Lambda 표현식으로 정의할 경우 `RunnableLambda(lambda 표현식)` 으로 정의해야 한다.
   - 보통 일반함수는 `RunnableLambda`를 사용할 필요 없다.
3. `@chain` decorator를 사용
   - 함수에 `@chain` decorator가 선언되면 그 함수는 `RunnableLambda` 타입이 된다.
   - 이 방식은 LCEL만으로 표현하기 어려운 실행 흐름을 직접 정의해야 할 때 주로 사용된다.
     - LCEL은 순차 실행구조를 따른다. 그래서  제어문을 이용해 그 흐름을 제어할 수가 없다. 
     - 단순한 파이프라인에서는 LCEL만으로도 충분하지만, 다음과 같은 경우에는 한계가 있다.
       - 특정 단계를 조건에 따라 실행하거나 생략해야 하는 경우
       - 동일한 단계를 반복적으로 실행해야 하는 경우
       - 여러 판단 로직에 따라 실행 경로가 달라지는 agent 구조
     - 이처럼 복잡한 업무 흐름을 가지는 agent는 단순한 순차 구조만으로는 원하는 응답 품질을 얻기 어렵다. 결국 실행 흐름 자체를 개발자가 직접 코드로 정의해야 하며, 이러한 경우에 `@chain`을 사용해 chain/agent 함수를 구현한다.
   - 이러한 복잡한 실행 흐름을 보다 구조적으로 정의하기 위해 LangChain에서 추가로 제공하는 것이 **LangGraph**이다.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def plus(n1, n2):
    return n1 + n2

def wrap_plus(x):
    return plus(x[0], x[1])

# chain = RunnablePassthrough() | RunnableLambda(lambda x: plus(x[0], x[1]))
chain = RunnablePassthrough() | wrap_plus
chain.invoke([1, 2])

3

In [ ]:
# recipe_chain과 translate_chain을 이용해 음식레시피를 특정 언어로 반환
### recipe_chain의 결과와 translate_chain의 결과 둘을 모두 반환
from langchain_core.runnables import chain
# RunnableLambda(multi_language_recipe_chain)

@chain
def multi_language_recipe_chain(input_data: dict) -> dict[str, str]:
    food: str = input_data['food'] # 음식 이름
    language: str = input_data['language'] # 레시피 언어
    is_korean: bool = input_data["is_korean"] # 한국어 레시피 필요 여부

    korean_recipe = recipe_chain.invoke({"food":food})

    result = translate_chain.invoke({"content": korean_recipe, "language": language})

    final_result = {"recipe":result}

    if is_korean: # 한국어 레시피도 요청
        final_result["korean_recipe"] = korean_recipe

    return final_result

multi_language_recipe_chain # @chain 없으면 그냥 function 

RunnableLambda(multi_language_recipe_chain)

In [35]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스", "language":"중국어", "is_korean":True}
)

In [36]:
res.keys()

dict_keys(['recipe', 'korean_recipe'])

In [38]:
print(res['korean_recipe'])
print(res['recipe'])

# 돈까스 레시피

## 요리 기본 정보
- **난이도:** 중
- **조리시간:** 30분~40분
- **인분:** 2인분

## 필요한 재료
### 주재료
- 돼지고기 등심 또는 안심 2장(약 300~400g)
- 소금 약간
- 후추 약간

### 튀김옷 재료
- 밀가루 1/2컵
- 달걀 2개
- 빵가루 1~1.5컵

### 튀김용
- 식용유 충분히

### 곁들임(선택)
- 양배추 채썬 것
- 밥
- 돈까스 소스

### 대체 재료
- **돼지고기 대신:** 닭가슴살로 치킨까스처럼 응용 가능
- **빵가루가 없을 때:** 식빵을 잘게 갈아 사용 가능
- **밀가루가 없을 때:** 튀김가루로 대체 가능

## 요리 방법
1. **고기 손질하기**  
   돼지고기 등심 또는 안심은 키친타월로 물기를 닦아줍니다.  
   칼등이나 고기망치로 가볍게 두드려 두께를 고르게 만들어 주세요.

2. **간하기**  
   고기 앞뒤에 소금과 후추를 살짝 뿌려 밑간합니다.

3. **튀김옷 입히기**  
   고기에 순서대로 **밀가루 → 달걀물 → 빵가루**를 묻혀줍니다.  
   빵가루는 손으로 살짝 눌러 잘 붙도록 해주세요.

4. **기름 예열하기**  
   팬에 식용유를 넉넉히 붓고 중불에서 예열합니다.  
   튀김 온도는 약 **170~180℃** 정도가 적당합니다.

5. **튀기기**  
   돈까스를 넣고 한 면당 2~3분씩 노릇하게 튀겨줍니다.  
   너무 센 불은 겉만 타고 속이 익지 않을 수 있으니 주의하세요.

6. **기름 빼기**  
   튀긴 돈까스는 키친타월이나 망 위에 올려 기름을 빼줍니다.

7. **서빙하기**  
   먹기 좋게 썰어 접시에 담고, 양배추채와 돈까스 소스를 곁들여 완성합니다.

## 팁
- 고기를 너무 두껍게 두드리면 튀길 때 퍽퍽해질 수 있으니 적당히 펴주세요.
- 빵가루를 입힌 뒤 5분 정도 두면 튀김옷이 더 잘 붙습니다.
- 한 번에 너무 많은 돈까스를 넣으면 기름 온도가 떨어져 바삭함이 줄어듭니다.
-

# Cache

- 응답 결과를 저장해서 같은 질문이 들어오면 LLM에 요청하지 않고 저장된 결과를 보여주도록 한다.
    - 처리속도와 비용을 절감할 수 있다.
    - 특히 chatbot같이 비슷한 질문을 하는 경우 유용하다.
- 저장 방식은 `메모리`, `sqlite` 등 다양한 방식을 지원한다.
  
    ```python
    set_llm_cache(Cache객체)
    ```

In [ ]:
# uv pip install langchain-community

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache

# Cache 설정은 한 번만 하면 됨
# set_llm_cache(InMemoryCache())
set_llm_cache(SQLiteCache("cache.sqlite")) # cache를 저장할 파일 경로

In [50]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스", "language":"중국어", "is_korean":True}
)

c:\Documents\SKN31\10_AI_Agent\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
c:\Documents\SKN31\10_AI_Agent\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]


In [51]:
print(res['korean_recipe'])

# 돈까스 레시피

## 요리 기본 정보
- **난이도**: 중
- **조리시간**: 약 30~40분
- **인분**: 2인분

## 필요한 재료
### 주재료
- 돼지고기 등심 또는 안심 2장(각 150~200g)
- 소금 약간
- 후추 약간
- 밀가루 1/2컵
- 달걀 2개
- 빵가루 1~2컵
- 식용유 적당량

### 선택 재료
- 양배추 채썬 것 약간
- 레몬 1/4개
- 돈까스 소스 적당량

### 대체 재료
- **돼지고기 대신**: 닭가슴살로도 만들 수 있음
- **빵가루가 없을 때**: 식빵을 잘게 부숴 사용 가능
- **튀김유가 없을 때**: 에어프라이어 또는 오븐 조리 가능

## 요리 방법
1. **고기 손질하기**  
   돼지고기는 두드려서 두께를 고르게 펴줍니다. 너무 두꺼우면 익히는 시간이 오래 걸리므로 1cm 정도가 적당합니다.

2. **간하기**  
   고기 양면에 소금과 후추를 살짝 뿌려 5분 정도 둡니다.

3. **튀김옷 준비하기**  
   볼 3개를 준비해 각각 밀가루, 풀어둔 달걀, 빵가루를 담습니다.

4. **튀김옷 입히기**  
   고기에 밀가루를 얇게 묻힌 뒤, 달걀물을 입히고 마지막으로 빵가루를 골고루 묻힙니다.  
   빵가루가 잘 붙도록 살짝 눌러줍니다.

5. **튀기기**  
   팬에 식용유를 넉넉히 붓고 중불로 예열합니다.  
   빵가루를 조금 떨어뜨렸을 때 바로 올라오면 온도가 적당합니다.  
   돈까스를 넣고 앞뒤로 노릇하게 튀겨줍니다.  
   한 번에 오래 튀기기보다 색이 나면 뒤집어 고르게 익힙니다.

6. **기름 빼기**  
   튀긴 돈까스는 키친타월이나 철망 위에 올려 기름을 빼줍니다.

7. **곁들이기**  
   먹기 좋게 썰어 양배추 채와 함께 담고, 돈까스 소스를 곁들입니다.

## 팁
- 고기를 너무 세게 두드리면 식감이 질겨질 수 있으니 적당히만 펴주세요.
- 빵가루를 입힌 뒤 5분 정도 두면 튀김옷이 더 잘 붙습니다.
- 기름 온도가 너무 낮으면 기름을 많이 먹

In [42]:
print(res['recipe'])

# 日式炸猪排食谱

## 基本信息
- **难度：** 中
- **烹饪时间：** 约 30～40 分钟
- **份量：** 2 人份

## 食材
### 主材料
- 猪里脊或猪里脊肉排 2 片（约 300～400g）
- 盐 1/2 小匙
- 胡椒 少许

### 裹粉材料
- 面粉 1/2 杯
- 鸡蛋 2 个
- 面包糠 1～1.5 杯

### 炸制用
- 食用油 适量

### 可选材料
- 高丽菜丝 一把
- 日式炸猪排酱 适量
- 米饭 2 碗

## 做法
1. **处理猪肉**  
   先用厨房纸巾把猪肉表面的水分擦干，再均匀撒上盐和胡椒调味。  
   如果肉块比较厚，可以用肉锤或刀背轻轻拍打，让厚度更均匀。

2. **裹上炸衣**  
   按照 **面粉 → 打散的鸡蛋 → 面包糠** 的顺序，依次将猪肉裹好。  
   面包糠可用手轻轻按压，让其更牢固地附着。

3. **预热油锅**  
   在平底锅中倒入足量食用油，加热至 **170～180℃**。  
   如果放入少许面包糠后能立刻浮起，就表示温度合适。

4. **炸猪排**  
   放入猪排后，每面炸 **2～4 分钟**，炸至金黄。  
   不要用太大的火，中火慢慢炸，这样内部能熟透，外层也会更酥脆。

5. **沥油**  
   炸好的猪排放在厨房纸巾或网架上沥油。  
   不要立刻切开，静置 1～2 分钟，可以减少肉汁流失。

6. **完成摆盘**  
   切成方便入口的大小后装盘，搭配高丽菜丝、米饭和炸猪排酱一起食用即可。

## 小贴士
- **想要更酥脆的话**，建议使用比普通面包糠更粗的 **生面包糠（panko）**。
- **想让肉质不干柴**，关键是不要炸太久。
- **也可以用空气炸锅**：在表面轻刷一层油后，以 180℃ 烹调 10～15 分钟即可。
- **如果没有炸猪排酱**，可以把番茄酱、伍斯特酱和糖混合，做成简易酱汁。

如果需要，我也可以继续为您整理一份**日式炸猪排酱食谱**。
